In [18]:
!pip install -q mediapipe opencv-python-headless

In [19]:
import os
import urllib.request

MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
MODEL_PATH = "/content/hand_landmarker.task"

urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

size = os.path.getsize(MODEL_PATH)
print(f"Downloaded {size / 1e6:.2f} MB")
assert size > 1_000_000, "Download failed or incomplete — file too small!"
print("Model downloaded successfully")

Downloaded 7.82 MB
Model downloaded successfully


In [29]:
from IPython.display import HTML, display

live_hand_tracker = """
<div style="text-align:center;">
  <video id="webcam" autoplay playsinline style="display:none;"></video>
  <canvas id="output_canvas" style="border:1px solid #444; max-width:100%;"></canvas>
  <h2 id="status" style="font-family:sans-serif; color:#0f0;">Loading model...</h2>
</div>

<script type="module">
import { HandLandmarker, FilesetResolver } from "https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@0.10.14";

const HAND_CONNECTIONS = [
  [0,1],[1,2],[2,3],[3,4],
  [0,5],[5,6],[6,7],[7,8],
  [5,9],[9,10],[10,11],[11,12],
  [9,13],[13,14],[14,15],[15,16],
  [13,17],[17,18],[18,19],[19,20],
  [0,17]
];

const video = document.getElementById("webcam");
const canvas = document.getElementById("output_canvas");
const ctx = canvas.getContext("2d");
const status = document.getElementById("status");

function distance(a, b) {
  return Math.hypot(a.x - b.x, a.y - b.y);
}

function countFingers(landmarks) {
  const wrist = landmarks[0];
  const pinkyMcp = landmarks[17];

  // Scale-normalize using palm size, so the threshold works at any distance from camera
  const palmSize = distance(wrist, landmarks[9]); // wrist to middle-finger MCP

  let count = 0;

  // Index, Middle, Ring, Pinky — tip farther from wrist than pip joint = extended
  const fingers = [
    { tip: 8,  pip: 6 },
    { tip: 12, pip: 10 },
    { tip: 16, pip: 14 },
    { tip: 20, pip: 18 }
  ];

  for (const f of fingers) {
    if (distance(wrist, landmarks[f.tip]) > distance(wrist, landmarks[f.pip]) + palmSize * 0.05) {
      count++;
    }
  }

  // Thumb: compare tip-to-pinky-MCP distance vs thumb-MCP-to-pinky-MCP distance
  // Extended thumb = far from the pinky side of the hand
  // Folded thumb = tucked close to the pinky side of the hand
  if (distance(landmarks[4], pinkyMcp) > distance(landmarks[2], pinkyMcp) + palmSize * 0.1) {
    count++;
  }

  return count;
}

async function main() {
  const vision = await FilesetResolver.forVisionTasks(
    "https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@0.10.14/wasm"
  );
  const handLandmarker = await HandLandmarker.createFromOptions(vision, {
    baseOptions: {
      modelAssetPath: "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
      delegate: "GPU"
    },
    runningMode: "VIDEO",
    numHands: 4
  });

  status.textContent = "Requesting camera access...";
  const stream = await navigator.mediaDevices.getUserMedia({ video: true });
  video.srcObject = stream;
  await video.play();

  canvas.width = video.videoWidth;
  canvas.height = video.videoHeight;
  status.textContent = "Live!";

  let lastTime = -1;
  function renderLoop() {
    if (video.currentTime !== lastTime) {
      lastTime = video.currentTime;
      const result = handLandmarker.detectForVideo(video, performance.now());

      ctx.save();
      ctx.clearRect(0, 0, canvas.width, canvas.height);
      ctx.drawImage(video, 0, 0, canvas.width, canvas.height);

      let totalFingers = 0;
      const numHands = result.landmarks.length;

      for (const landmarks of result.landmarks) {
        // Draw connections
        ctx.strokeStyle = "#00FF00";
        ctx.lineWidth = 3;
        for (const [a, b] of HAND_CONNECTIONS) {
          ctx.beginPath();
          ctx.moveTo(landmarks[a].x * canvas.width, landmarks[a].y * canvas.height);
          ctx.lineTo(landmarks[b].x * canvas.width, landmarks[b].y * canvas.height);
          ctx.stroke();
        }
        // Draw dots
        ctx.fillStyle = "#FF0000";
        for (const lm of landmarks) {
          ctx.beginPath();
          ctx.arc(lm.x * canvas.width, lm.y * canvas.height, 4, 0, 2 * Math.PI);
          ctx.fill();
        }
        totalFingers += countFingers(landmarks);
      }

      ctx.font = "28px sans-serif";
      ctx.fillStyle = "#700707";
      ctx.fillText(`Hands: ${numHands}  |  Fingers: ${totalFingers}`, 10, 30);
      ctx.restore();

      status.textContent = `Hands: ${numHands} | Fingers up: ${totalFingers}`;
    }
    requestAnimationFrame(renderLoop);
  }
  renderLoop();
}

main();
</script>
"""

display(HTML(live_hand_tracker))